In [1]:
import torch
import torch.nn as nn
import numpy as np

In [2]:
def forward_diffusion_process(x_0, t, beta_schedule):
    """
    x_0: Original image
    t: Time step
    beta_schedule: Noise schedule
    """
    
    # Calculate cumulative noise
    alpha_bar_t = torch.prod(1 - beta_schedule[:t+1])
    
    # Sample noise
    noise = torch.rand_like(x_0)
    
    # Add noise
    x_t = torch.sqrt(alpha_bar_t) * x_0 + torch.sqrt(1 - alpha_bar_t) * noise
    
    return x_t, noise

In [ ]:
def reverse_diffusion_step(x_t, t, model, beta_schedule):
    # Predict noise
    predicted_noise = model(x_t, t)
    
    # Calculate parameters
    alpha_t = 1 - beta_schedule[t]
    alpha_bar_t = torch.prod(1 - beta_schedule[:t+1])
    alpha_bar_t_prev = torch.prod(1 - beta_schedule[:t])
    
    # Denoise
    pred_x_0 = (x_t - torch.sqrt(1 - alpha_bar_t) * predicted_noise) / torch.sqrt(alpha_bar_t)
    
    # Sample next step
    posterior_variance = beta_schedule[t] * (1 - alpha_bar_t_prev) / (1 - alpha_bar_t)
    x_t_prev = torch.sqrt(alpha_bar_t_prev) * pred_x_0 + torch.sqrt(posterior_variance) * torch.randn_like(x_t)
    
    return x_t_prev